To install [`minepy`](https://github.com/minepy/minepy) on newer python ([courtesy of user WMF1997](https://github.com/minepy/minepy/issues/45)):
```bash
git clone https://github.com/minepy/minepy
rm minepy/mine.c 
./compile_pyx.sh
python3 setup.py build_ext --inplace 
python3 setup.py install
```
NB: you must have your correct virtual environment activated, `cython` installed and `setuptools` updated

In [191]:
import torch
from dinosaw.models.vit_wrapper import MODEL_LIST, PretrainedViTWrapper, AlibiVitWrapper
from dinosaw.comaprisons.denoising_vits import DenoisingViTWrapper 
from dinosaw.utils import do_2D_pca, to_numpy, closest_resize, convert_image

import numpy as np
from minepy import MINE
from PIL import Image



SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

In [192]:
S = 14
dv2_model = PretrainedViTWrapper(
    MODEL_LIST[1],
    stride=S,
    add_flash_attn=False,
    device="cuda:0",
)

dv2_model.eval()
None

In [193]:
dvt_model = DenoisingViTWrapper(
    '../../trained_models/dvt.pth',
    MODEL_LIST[1],
    stride=S,
    add_flash_attn=False,
    device="cuda:0",
)
dvt_model.eval()
None

In [194]:

alibi_model = AlibiVitWrapper(
    MODEL_LIST[1],
    stride=S,
    add_flash_attn=False,
    device="cuda:0",
    slope_type="constant",
    normalize=True,
    wrap=True,
)

weights = torch.load("../../trained_models/alibi_homog_dv2_vits14_reg.pth", weights_only=True, map_location="cuda:0")
alibi_model.load_state_dict(weights)
alibi_model.eval()
# print(alibi_model.model.pos_embed)
None

In [195]:
def get_features(model: PretrainedViTWrapper, pil_img: Image.Image, channel_last: bool=True) -> np.ndarray:
    tr = closest_resize(pil_img.height, pil_img.width, 14)
    img_tensor = convert_image(pil_img, tr, device_str="cuda:0", to_half=False)
    with torch.no_grad():
        emb = model.forward_features(img_tensor, make_2D=True)
    emb_np = to_numpy(emb.squeeze(0))
    if channel_last:
        emb_np = np.transpose(emb_np, (1, 2, 0))
    return emb_np

In [196]:
def get_mic(sample_0: np.ndarray, sample_1: np.ndarray) -> float:
    mine = MINE(alpha=0.6, c=15)
    mine.compute_score(sample_0, sample_1)
    mic = mine.mic()
    return mic


def get_positional_mics(feats: np.ndarray) -> tuple[list[float], list[float]]:
    h, w, c = feats.shape
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))

    xx  = xx / float(w-1)
    yy = yy / float(h-1 )
    feats_flat = feats.reshape(h * w, c)

    mic_xs, mic_ys = [], []
    for i in range(c):
        feat_channel = feats_flat[:, i]
        mic_x = get_mic(feat_channel, xx.flatten())
        mic_y = get_mic(feat_channel, yy.flatten())
        mic_xs.append(mic_x)
        mic_ys.append(mic_y)
    return mic_xs, mic_ys

def get_mic_stats(mic_xs: list[float], mic_ys: list[float]) -> tuple[float, float, float]:
    max_mic = (np.max(mic_xs) + np.max(mic_ys)) / 2
    mean_mic = (np.mean(mic_xs) + np.mean(mic_ys)) / 2
    std_mic = (np.std(mic_xs) + np.std(mic_ys)) / 2
    return float(max_mic), float(mean_mic), float(std_mic)

In [197]:
# img = Image.open("../../images/micro/diff_shapes_518.png").convert("RGB")
img_fname = 'bulldog_518.png'
img = Image.open(f"data/MIC/{img_fname}").convert("RGB")
tr = closest_resize(518, 518, 14)
img_tensor = convert_image(img, tr, device_str="cuda:0", to_half=False)

dv2_feats = get_features(dv2_model, img)
dvt_feats = get_features(dvt_model, img)
alibi_feats = get_features(alibi_model, img)

In [199]:
dv2_mic_xs, dv2_mic_ys = get_positional_mics(dv2_feats)

In [200]:

dvt_mic_xs, dvt_mic_ys = get_positional_mics(dvt_feats)

In [201]:

alibi_mic_xs, alibi_mic_ys = get_positional_mics(alibi_feats)

In [202]:
%%capture
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec



feat_map = {'DINOv2': dv2_feats, 'DVT': dvt_feats, 'Alibi-Dv2_H': alibi_feats}
mic_map = {'DINOv2': [dv2_mic_xs, dv2_mic_ys],
           'DVT': [dvt_mic_xs, dvt_mic_ys], 
           'Alibi-Dv2_H': [alibi_mic_xs, alibi_mic_ys]}

which = 'Alibi-Dv2_H'


mics = [dvt_mic_xs, dvt_mic_ys]
# mics = [alibi_mic_xs, alibi_mic_ys]

FS = 20
fig = plt.figure(figsize=(30,10))
gs = GridSpec(2, 6, )

plt.suptitle(f'MIC Analysis - {which}, {img_fname}', fontsize=FS + 6)

img_ax = fig.add_subplot(gs[0, 0])
img_ax.imshow(img)
img_ax.axis('off')


dvt_reduced = do_2D_pca(feat_map[which].transpose(2, 0, 1), 3, pre_norm='std', post_norm='minmax')

pca_ax = fig.add_subplot(gs[1, 0])
pca_ax.imshow(dvt_reduced)
pca_ax.axis('off')



colours = ['red', 'blue','green']

for i, xs in enumerate([dv2_mic_xs, dvt_mic_xs, alibi_mic_xs]):
    x_plot_ax = fig.add_subplot(gs[0, i + 1])

    if i == 0:
        x_plot_ax.set_ylabel('MIC with X Position', fontsize=FS - 2)

    x_plot_ax.plot(xs, color=colours[i])
    x_plot_ax.set_title(list(feat_map.keys())[i], fontsize=FS)
    x_plot_ax.set_ylim(0, 1)

for j, ys in enumerate([dv2_mic_ys, dvt_mic_ys, alibi_mic_ys]):
    y_plot_ax = fig.add_subplot(gs[1, j + 1])

    if j == 0:
        y_plot_ax.set_ylabel('MIC with Y Position', fontsize=FS - 2)

    y_plot_ax.plot(ys, color=colours[j])
    y_plot_ax.set_ylim(0, 1)



dirs = ['x', 'y']
for j, ys in enumerate(mic_map[which]):
    worst_ax = fig.add_subplot(gs[j, 4])

    worst_ch_mic = np.max(ys)
    worst_ch_idx = np.argmax(ys)
    worst_ch = feat_map[which][:, :, worst_ch_idx]


    worst_ax.set_title(f'{which} ch: {worst_ch_idx}, {dirs[j]}-MIC: {worst_ch_mic:.3f}', fontsize=FS - 2)

    worst_ax.imshow(worst_ch)
    worst_ax.set_axis_off()



plt.tight_layout(pad=1.2)